In [1]:
#include <vector>
#include <iostream>
#include <sstream>

The ***mapper*** class allows you to link an arbitrary *algorithm* for element-by-element processing of a data set (the *algorithm* inherits the ***mapper*** class) and an arbitrary implementation of the *mechanism* for performing element-by-element processing (the *mechanism* for performing element-by-element processing inherits the ***mapper::engine*** class).

In [2]:
class mapper{
public:
    class engine{
    friend class mapper;
    protected:
        virtual void init(unsigned size) = 0;
        virtual void map() = 0;
    protected:
        void on_init(unsigned size){_mapper->on_init(size);}
        void on_map(unsigned id){_mapper->on_map(id);}
    	void on_save(unsigned id, std::ostream&out, bool mapped){
            _mapper->on_save(id,out,mapped);
        }
    	void on_load(unsigned id, std::istream&in, bool mapped){
            _mapper->on_load(id,in,mapped);
        }
    protected:
        mapper* _mapper;
    };
public:
    mapper(mapper::engine& eng):_engine(eng){eng._mapper=this;}
    void map(){_engine.map();}
    void init(unsigned size){_engine.init(size);}
protected:
    virtual void on_init(unsigned size) = 0;
    virtual void on_map(unsigned id) = 0;
	virtual void on_save(unsigned id, std::ostream&, bool mapped) = 0;
	virtual void on_load(unsigned id, std::istream&, bool mapped) = 0;
private:
    mapper::engine& _engine;
};

The ***basic_engine*** class illustrates a basic sequential implementation of element-by-element processing of an arbitrary data array. In addition to calling the *on_map(id)* method for processing an array element, a test uploading/downloading of raw elements  (*on_save(id,sstr,false); on_load(id,sstr,false)*); and processed array elements are performed (*on_save(id,sstr,true); on_load(id,sstr,true);*).

In [3]:
class basic_engine: public mapper::engine{
    void init(unsigned size)override
            {_size=size;on_init(size);}
    void map()override{
        for(int id=0;id<_size;id++){
            {   std::stringstream sstr;
                on_save(id,sstr,false);
                on_load(id,sstr,false);
            }
            on_map(id);
            {   std::stringstream sstr;
                on_save(id,sstr,true);
                on_load(id,sstr,true);            
            }
        }
    }
    unsigned _size;
};
class sync_state_engine: public mapper::engine{};
class everest_engine: public mapper::engine{};
//...

The ***throughput_test_mapper*** test squares each element of the original *N*-array. The result is written to an *NxN* array. Since the operation takes little time, the test allows us to estimate the overhead of running an operation to process an element of the data array. Additionally, the test verifies the correctness of the *algorithm's* implementation and execution *mechanism*.

In [4]:
class throughput_test_mapper: public mapper{
public:
    throughput_test_mapper(mapper::engine& eng): mapper(eng){}
private:
    void on_init(unsigned size) override{
        N.resize(size); NxN.resize(size);
    }
    void on_map(unsigned id) override{
        NxN[id] = N[id] * N[id];
    }
    void on_save(unsigned id, std::ostream& out, bool mapped) override{
        if(mapped) out << NxN[id]; else out << N[id];  
    }
	void on_load(unsigned id, std::istream& in, bool mapped) override{
        if(mapped) in >> NxN[id]; else in >> N[id];
    }
public:
    std::vector<int> N;
    std::vector<int> NxN;
};
class compute_intensive_test_mapper: public mapper{};
class data_intensive_test_mapper: public mapper{};
//...

The method of testing multiple implementations (***basic_engine***, ***sync_state_engine***, ***everest_engine***) of the algorithm for element-by-element squaring of numeric array elements. The *if(master){..}* sections for input data and output of processing results are executed in the main process (*master==true*).

In [5]:
{
    const int SIZE = 10;
    bool master = true;
    
    basic_engine eng;
    //sync_state_engine eng;
    //everest_engine eng;
    throughput_test_mapper a_mapper(eng);

    if(master){
        a_mapper.init(SIZE);
        for(int i=0;i<SIZE;i++) a_mapper.N[i] = i;
    }
    a_mapper.map();

    if(master){
        for(int i=0;i<SIZE;i++)
            std::cout << a_mapper.N[i] <<"^2 = " << a_mapper.NxN[i] << std::endl;
    }
}

0^2 = 0
1^2 = 1
2^2 = 4
3^2 = 9
4^2 = 16
5^2 = 25
6^2 = 36
7^2 = 49
8^2 = 64
9^2 = 81
